# 第七讲：RAG 进阶与 Agent 记忆系统

**受众**：已完成第6讲学习的 Python 进阶课学生
**前置条件**：
- 第6讲已搭建基础 RAG 流水线（ChromaDB + sentence-transformers）
- 已有 `./chroma_db` 向量库可用

**学习目标**：
1. 实现 MQE / HyDE / Reranker 高级检索策略
2. 掌握 RAG 评估指标（Recall@k / MRR）
3. 实现简化版 Agent 记忆系统


In [ ]:
# 环境准备
%pip install -q rank-bm25 jieba sentence-transformers chromadb openai

# 复用第6讲的配置
from sentence_transformers import SentenceTransformer
import chromadb

embed_model = SentenceTransformer('all-MiniLM-L6-v2')
client = chromadb.PersistentClient(path='./chroma_db')
collection = client.get_collection('paper_qa')
print(f'已有 {collection.count()} 条向量数据')


## Part 1：高级检索策略

**RAG 检索的三大痛点**：
1. **用词不匹配**：用户问"法国出口额"，文档写"法兰西出口"
2. **查询-文档语义鸿沟**：问题是疑问句，文档是陈述句
3. **top-k 中间丢失**（lost in the middle）：LLM 对中间位置关注度不够

**三种高级策略**：
- **MQE**（多查询扩展）：解决用词多样性
- **HyDE**（假设文档嵌入）：解决语义鸿沟
- **Reranker**（重排序）：解决中间丢失

### 1.1 策略1：多查询扩展 MQE

**思想**：用 LLM 把原查询改写成 N 个语义等价的查询，并行检索后合并。

```
原查询："法国出口额"
    ↓ LLM 扩展
["法兰西出口", "对法出口贸易额", "法国出口数据"]
    ↓ 并行检索
三个 top-5 结果合并去重
    ↓ 取 top-5
更全面的召回
```

**优势**：覆盖用词多样性，显著提升召回率。

In [19]:
def expand_query_mqe(query, n=3):
    """用 LLM 生成多样化查询扩展"""
    prompt = f"""你是检索查询扩展助手。生成语义等价或互补的多样化查询。
原始查询：{query}
请给出 {n} 个不同表述的查询，每行一个，用中文，简短，避免标点。"""

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=100,
            temperature=0.7,  # 高温度增加多样性
        )
        text = response.choices[0].message.content
        lines = [ln.strip("- •\t123. ") for ln in text.splitlines() if ln.strip()]
        return lines[:n] or [query]
    except Exception:
        return [query]

# 测试 MQE
query = "2024 年商业航天发射情况"
expansions = expand_query_mqe(query, n=3)
print(f"原查询：{query}")
print(f"\n扩展查询：")
for i, e in enumerate(expansions):
    print(f"  {i+1}. {e}")


原查询：2024 年商业航天发射情况

扩展查询：
  1. 024年商业火箭发射预测
  2. 024商业航天活动概述
  3. 未来一年商业太空任务分析


In [20]:
def retrieve_mqe(query, top_k=5, n_expansions=3):
    """MQE 检索：多查询扩展 + 合并去重"""
    # 生成扩展查询
    expansions = expand_query_mqe(query, n=n_expansions)
    all_queries = [query] + expansions

    # 并行检索，合并结果
    all_results = {}
    for q in all_queries:
        results = retrieve(q, top_k=top_k)
        for r in results:
            chunk_id = r["metadata"]["chunk_idx"]
            if chunk_id not in all_results or r["score"] > all_results[chunk_id]["score"]:
                all_results[chunk_id] = r

    # 按分数排序，取 top_k
    merged = sorted(all_results.values(), key=lambda x: x["score"], reverse=True)
    return merged[:top_k]

# 对比基础检索 vs MQE
query = "2024 年商业航天发射情况"
print("【基础检索 top-5】")
base_results = retrieve(query, top_k=5)
for i, r in enumerate(base_results):
    print(f"  [{i+1}] score={r['score']:.3f} chunk_{r['metadata']['chunk_idx']}")

print(f"\n【MQE 检索 top-5】")
mqe_results = retrieve_mqe(query, top_k=5)
for i, r in enumerate(mqe_results):
    print(f"  [{i+1}] score={r['score']:.3f} chunk_{r['metadata']['chunk_idx']}")

# 看是否召回更多不同 chunk
base_ids = {r["metadata"]["chunk_idx"] for r in base_results}
mqe_ids = {r["metadata"]["chunk_idx"] for r in mqe_results}
print(f"\n基础检索召回 chunk：{sorted(base_ids)}")
print(f"MQE 召回 chunk：{sorted(mqe_ids)}")
print(f"MQE 新增召回：{sorted(mqe_ids - base_ids)}")


【基础检索 top-5】
  [1] score=0.186 chunk_4
  [2] score=0.155 chunk_13
  [3] score=0.105 chunk_6
  [4] score=0.100 chunk_3
  [5] score=0.060 chunk_5

【MQE 检索 top-5】
  [1] score=0.190 chunk_4
  [2] score=0.155 chunk_13
  [3] score=0.136 chunk_6
  [4] score=0.124 chunk_3
  [5] score=0.119 chunk_7

基础检索召回 chunk：[3, 4, 5, 6, 13]
MQE 召回 chunk：[3, 4, 6, 7, 13]
MQE 新增召回：[7]


### 1.2 策略2：假设文档嵌入 HyDE

**思想**：用答案找答案——先让 LLM 生成假设答案，用假设答案的向量去检索。

```
原查询："2024 年发射次数？"
    ↓ LLM 生成假设答案
"2024 年中国商业航天发射次数预计达 30 次，同比增加 50%..."
    ↓ 用假设答案向量化去检索
命中真实文档（陈述句对陈述句）
```

**为什么有效**：问题是疑问句，文档是陈述句，语义空间不同；假设答案也是陈述句，更接近真实文档。

**适合**：专业领域查询、术语密集场景。

### 1.3 策略3：重排序 Reranker

**为什么需要**：向量检索是 Bi-Encoder（双塔独立编码），精度有限。

| 架构 | 原理 | 速度 | 精度 | 用途 |
|---|---|---|---|---|
| Bi-Encoder | query 和 doc 独立编码，算余弦相似度 | 快 | 中 | 粗排（向量检索） |
| Cross-Encoder | query 和 doc 拼接后一起编码 | 慢 | 高 | 精排（Reranker） |

**流程**：向量检索 top-10（粗排）→ Reranker 精排 → 取 top-3

**解决 lost in the middle**：精排后只取 top-3，避免中间位置被忽略。

### 1.5 策略4：混合检索（了解）

**思想**：向量语义检索 + BM25 关键词检索，按分数加权融合。

```python
def hybrid_retrieve(query, top_k=5, alpha=0.5):
    # 向量检索
    vec_results = retrieve(query, top_k=10)
    # BM25 检索（略，可用 rank_bm25 库）
    bm25_results = bm25_search(query, top_k=10)
    # 分数融合
    merged = merge_scores(vec_results, bm25_results, alpha=alpha)
    return merged[:top_k]
```

**alpha 权重**：alpha=0.5 均衡 / 0.7 偏语义 / 0.3 偏关键词

> BM25 是 Naive RAG 的检索方式，向量是 Advanced RAG 的，混合是 Modular RAG 的典型做法。


### 1.4 RAG 评估指标

**如何量化检索质量？**

| 指标 | 定义 | 用途 |
|---|---|---|
| **Recall@k** | top-k 结果中包含正确答案的比例 | 召回率（找全了吗） |
| **Precision@k** | top-k 结果中相关的比例 | 精确率（找得准吗） |
| **MRR** | 第一个相关结果的平均倒数排名 | 排序质量 |


## Part 2：Agent 记忆系统

**LLM 的两个根本局限**：
1. **无状态**：跨会话遗忘（重启 Agent 就"忘了"用户）
2. **上下文窗口有限**：长对话早期信息丢失

**解决**：引入 Memory 系统，让 Agent 能"记住过去"。

**本 Part 用简化实现**（JSON + sentence-transformers，不依赖 hello-agents 框架），教学透明。

### 2.1 认知科学启发：人类记忆层次

```
人类记忆
├── 感觉记忆（0.5-3秒，容量巨大）
├── 工作记忆（15-30秒，7±2 项）
└── 长期记忆（终生，无限）
    ├── 程序性记忆（技能：骑自行车）
    └── 陈述性记忆（可表达）
        ├── 语义记忆（知识：巴黎是法国首都）
        └── 情景记忆（经历：昨天会议内容）
```

**启发**：Agent 也应该有分层记忆系统，而非单一消息列表。

### 2.2 Agent 四类型记忆

| 类型 | 存内容 | 生命周期 | 实现 | 类比 |
|---|---|---|---|---|
| **工作记忆** | 当前对话上下文 | 会话级，TTL 清理 | 内存 list | "现在在聊什么" |
| **情景记忆** | 具体交互事件 | 长期持久化 | JSON + 向量检索 | "上周问过 X" |
| **语义记忆** | 抽象知识/用户偏好 | 长期持久化 | JSON + 向量检索 | "用户偏好 Python" |
| **感知记忆** | 多模态信息 | 动态管理 | 文件路径 + 元数据 | "用户上传的图" |

**记忆五操作**：编码 → 存储 → 检索 → 整合（短期→长期）→ 遗忘

In [28]:
import json
from datetime import datetime, timedelta
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Optional

@dataclass
class MemoryItem:
    """记忆条目"""
    id: str
    content: str
    memory_type: str  # working / episodic / semantic / perceptual
    importance: float  # 0.0-1.0
    timestamp: str
    metadata: dict

# ============================================================
# 1. 工作记忆：内存 list + TTL 自动清理
# ============================================================
class WorkingMemory:
    """工作记忆：当前对话上下文，容量有限 + TTL"""

    def __init__(self, max_capacity=50, ttl_minutes=60):
        self.max_capacity = max_capacity
        self.ttl_minutes = ttl_minutes
        self.memories: list[MemoryItem] = []

    def add(self, content, importance=0.5, **metadata):
        # 过期清理
        self._expire_old()
        # 容量管理
        if len(self.memories) >= self.max_capacity:
            self._remove_lowest_priority()
        # 添加
        item = MemoryItem(
            id=f"wm_{datetime.now().strftime('%Y%m%d%H%M%S%f')}",
            content=content,
            memory_type="working",
            importance=importance,
            timestamp=datetime.now().isoformat(),
            metadata=metadata
        )
        self.memories.append(item)
        return item.id

    def retrieve(self, query, limit=5):
        self._expire_old()
        # 简单关键词匹配（工作记忆是临时的，不必向量检索）
        scored = []
        for m in self.memories:
            score = sum(1 for w in query.split() if w in m.content) / max(len(query.split()), 1)
            score *= (0.8 + m.importance * 0.4)  # 重要性加权
            if score > 0:
                scored.append((score, m))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [m for _, m in scored[:limit]]

    def _expire_old(self):
        cutoff = datetime.now() - timedelta(minutes=self.ttl_minutes)
        self.memories = [m for m in self.memories if datetime.fromisoformat(m.timestamp) > cutoff]

    def _remove_lowest_priority(self):
        self.memories.sort(key=lambda m: m.importance, reverse=True)
        self.memories = self.memories[:self.max_capacity - 1]

    def __len__(self):
        return len(self.memories)

# 测试工作记忆
wm = WorkingMemory(max_capacity=5, ttl_minutes=60)
wm.add("用户问了 2024 年发射次数", importance=0.8)
wm.add("用户偏好 Python 代码", importance=0.7)
wm.add("当前任务：分析航天报告", importance=0.9)
print(f"工作记忆条目数：{len(wm)}")
print(f"检索'用户'：{[m.content for m in wm.retrieve('用户')]}")


# ============================================================
# 2. 情景记忆 + 语义记忆：JSON 持久化 + 向量检索
# ============================================================
class LongTermMemory:
    """长期记忆：JSON 持久化 + 向量检索（情景/语义共用）"""

    def __init__(self, memory_type: str, storage_path: str, embed_model=None):
        self.memory_type = memory_type  # episodic / semantic
        self.storage_path = Path(storage_path)
        self.storage_path.parent.mkdir(parents=True, exist_ok=True)
        self.embed_model = embed_model
        self.memories: list[MemoryItem] = []
        self._load()

    def add(self, content, importance=0.5, **metadata):
        item = MemoryItem(
            id=f"{self.memory_type}_{datetime.now().strftime('%Y%m%d%H%M%S%f')}",
            content=content,
            memory_type=self.memory_type,
            importance=importance,
            timestamp=datetime.now().isoformat(),
            metadata=metadata
        )
        self.memories.append(item)
        self._save()
        return item.id

    def retrieve(self, query, limit=5):
        if not self.embed_model or not self.memories:
            # 退化为关键词匹配
            scored = [(sum(1 for w in query.split() if w in m.content), m) for m in self.memories]
            scored.sort(key=lambda x: x[0], reverse=True)
            return [m for s, m in scored[:limit] if s > 0]

        # 向量检索
        query_vec = self.embed_model.encode(query, normalize_embeddings=True)
        doc_vecs = self.embed_model.encode(
            [m.content for m in self.memories],
            normalize_embeddings=True
        )
        scores = (doc_vecs @ query_vec).tolist()

        # 重要性 + 时间衰减加权
        for i, m in enumerate(self.memories):
            age_days = (datetime.now() - datetime.fromisoformat(m.timestamp)).days
            time_decay = max(0.5, 1.0 - age_days * 0.01)  # 每天衰减 1%
            scores[i] *= time_decay * (0.8 + m.importance * 0.4)

        ranked = sorted(zip(scores, self.memories), key=lambda x: x[0], reverse=True)
        return [m for s, m in ranked[:limit]]

    def _load(self):
        if self.storage_path.exists():
            data = json.loads(self.storage_path.read_text(encoding="utf-8"))
            self.memories = [MemoryItem(**item) for item in data]

    def _save(self):
        data = [asdict(m) for m in self.memories]
        self.storage_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

    def __len__(self):
        return len(self.memories)

# 测试长期记忆
episodic = LongTermMemory("episodic", "./memory/episodic.json", embed_model)
semantic = LongTermMemory("semantic", "./memory/semantic.json", embed_model)

episodic.add("用户问了 2024 年商业航天发射次数", importance=0.8, event_type="question")
semantic.add("用户偏好带引用标注的回答", importance=0.9, category="preference")

print(f"情景记忆条目数：{len(episodic)}")
print(f"语义记忆条目数：{len(semantic)}")
print(f"\n检索'发射'：{[m.content for m in episodic.retrieve('发射')]}")


工作记忆条目数：3
检索'用户'：['用户问了 2024 年发射次数', '用户偏好 Python 代码']
情景记忆条目数：3
语义记忆条目数：3

检索'发射'：['用户上次问的是发射次数', '用户问了 2024 年商业航天发射次数', '用户问了 2024 年商业航天发射次数']


### 2.3 记忆五操作与遗忘策略

**五操作**：编码 → 存储 → 检索 → 整合（短期→长期）→ 遗忘

**三种遗忘策略**：
- 基于重要性：importance < 0.3 删除
- 基于时间：超过 30 天删除
- 基于容量：超过 1000 条删最不重要的

In [23]:
class MemoryManager:
    """记忆管理器：统一管理四类型记忆 + 遗忘策略"""

    def __init__(self, embed_model=None):
        self.working = WorkingMemory(max_capacity=50, ttl_minutes=60)
        self.episodic = LongTermMemory("episodic", "./memory/episodic.json", embed_model)
        self.semantic = LongTermMemory("semantic", "./memory/semantic.json", embed_model)
        # perceptual 暂不实现（多模态，需额外依赖）

    def add(self, content, memory_type="working", importance=0.5, **metadata):
        """编码 + 存储"""
        if memory_type == "working":
            return self.working.add(content, importance, **metadata)
        elif memory_type == "episodic":
            return self.episodic.add(content, importance, **metadata)
        elif memory_type == "semantic":
            return self.semantic.add(content, importance, **metadata)
        else:
            raise ValueError(f"未知记忆类型：{memory_type}")

    def search(self, query, memory_type=None, limit=5):
        """检索"""
        results = []
        if memory_type is None or memory_type == "working":
            results.extend(self.working.retrieve(query, limit))
        if memory_type is None or memory_type == "episodic":
            results.extend(self.episodic.retrieve(query, limit))
        if memory_type is None or memory_type == "semantic":
            results.extend(self.semantic.retrieve(query, limit))
        return results

    def consolidate(self, importance_threshold=0.7):
        """整合：把重要的工作记忆转为长期记忆"""
        moved = 0
        for m in self.working.memories[:]:
            if m.importance >= importance_threshold:
                self.episodic.add(m.content, m.importance, **m.metadata)
                self.working.memories.remove(m)
                moved += 1
        return f"整合了 {moved} 条工作记忆 → 情景记忆"

    def forget(self, strategy="importance", **kwargs):
        """遗忘策略"""
        if strategy == "importance":
            threshold = kwargs.get("threshold", 0.3)
            before = len(self.episodic) + len(self.semantic)
            self.episodic.memories = [m for m in self.episodic.memories if m.importance >= threshold]
            self.semantic.memories = [m for m in self.semantic.memories if m.importance >= threshold]
            after = len(self.episodic) + len(self.semantic)
            self.episodic._save()
            self.semantic._save()
            return f"遗忘 {before - after} 条低重要性记忆（threshold={threshold}）"

        elif strategy == "time":
            days = kwargs.get("days", 30)
            cutoff = datetime.now() - timedelta(days=days)
            before = len(self.episodic)
            self.episodic.memories = [
                m for m in self.episodic.memories
                if datetime.fromisoformat(m.timestamp) > cutoff
            ]
            self.episodic._save()
            return f"遗忘 {before - len(self.episodic)} 条超过 {days} 天的情景记忆"

        elif strategy == "capacity":
            max_size = kwargs.get("max_size", 1000)
            for store in [self.episodic, self.semantic]:
                if len(store) > max_size:
                    store.memories.sort(key=lambda m: m.importance, reverse=True)
                    store.memories = store.memories[:max_size]
                    store._save()
            return f"容量限制到 {max_size} 条"

        return "未知策略"

# 测试
mgr = MemoryManager(embed_model)
mgr.add("用户问了长征十二号", memory_type="working", importance=0.6)
mgr.add("用户上次问的是发射次数", memory_type="episodic", importance=0.8)
mgr.add("用户偏好简洁回答", memory_type="semantic", importance=0.9)

print("检索'用户'：")
for m in mgr.search("用户"):
    print(f"  [{m.memory_type}] {m.content}")


# 测试遗忘策略
print("=== 遗忘策略测试 ===\n")

# 添加一些低重要性记忆
mgr.add("临时记录1", memory_type="working", importance=0.1)
mgr.add("临时记录2", memory_type="episodic", importance=0.2)

print(f"遗忘前：情景记忆 {len(mgr.episodic)} 条，语义记忆 {len(mgr.semantic)} 条")
result = mgr.forget(strategy="importance", threshold=0.3)
print(f"{result}")
print(f"遗忘后：情景记忆 {len(mgr.episodic)} 条，语义记忆 {len(mgr.semantic)} 条")

# 测试整合
print(f"\n=== 整合测试 ===")
print(f"整合前：工作记忆 {len(mgr.working)} 条，情景记忆 {len(mgr.episodic)} 条")
result = mgr.consolidate(importance_threshold=0.7)
print(result)
print(f"整合后：工作记忆 {len(mgr.working)} 条，情景记忆 {len(mgr.episodic)} 条")


检索'用户'：
  [working] 用户问了长征十二号
  [episodic] 用户上次问的是发射次数
  [episodic] 用户问了 2024 年商业航天发射次数
  [semantic] 用户偏好简洁回答
  [semantic] 用户偏好带引用标注的回答
=== 遗忘策略测试 ===

遗忘前：情景记忆 3 条，语义记忆 2 条
遗忘 1 条低重要性记忆（threshold=0.3）
遗忘后：情景记忆 2 条，语义记忆 2 条

=== 整合测试 ===
整合前：工作记忆 2 条，情景记忆 2 条
整合了 0 条工作记忆 → 情景记忆
整合后：工作记忆 2 条，情景记忆 2 条


### 2.4 Memory 与 RAG 的关系

| 维度 | RAG | Memory |
|---|---|---|
| 检索对象 | 外部知识库（文档） | 内部交互历史（对话） |
| 数据来源 | 用户上传的文档 | Agent 与用户的交互 |
| 时效性 | 静态（建库后不变） | 动态（每次交互都在增长） |
| 用途 | 注入领域知识 | 个性化、上下文延续 |

**结合方式**：检索 Memory（找历史偏好） + 检索 RAG（找知识）→ 拼 Prompt → LLM 生成 → 把本次交互存入 Memory

**Vibe Coding 视角**：Memory 是上下文工程的升级版——从"临时塞上下文"到"持久化、可检索、可遗忘"。

| 层级 | 第1讲做法 | 第5讲升级 |
|---|---|---|
| 系统级 | Rules 文件 | Rules + 长期语义记忆 |
| 会话级 | 临时对话历史 | 工作记忆（TTL 清理） |
| 任务级 | @文件引用 | 情景记忆（跨会话持久化） |
| 引用级 | 单次 @引用 | Memory + RAG 联合检索 |